# CS-781 Phase 6 - Abstractive Legal Summarisation on our own Phase-1 models
**Settings:** Accelerator **GPU T4 x2**, Internet **On**, secret **HF_TOKEN** switched ON. Then run the cells top to bottom
(or **Save Version -> Save & Run All**).

Everything is uploaded to `<you>/cs781-p1-ckpt/phase6/`; finished runs are skipped if you re-run.
1. Cell 1 - installs the metric libraries of the official evaluator (`rouge-score`, `bert-score`), logs in, downloads our tokenizer.
2. Cell 3 - downloads the official `Phase_6_Summ` data once, cleans + splits documents into sentences + tokenises them with our
   tokenizer, holds out **200 training documents as a validation split** (the task has no dev set), caches everything on HF.
3. Cell 7 - safety test on the real data (3 training steps, validation summaries scored by the OFFICIAL evaluator, full test
   prediction + zip), then 11 training runs + 7 decoding runs on both GPUs (about 4 h):
   all four Phase-1 models; LoRMA (compulsory), LoRA, Houlsby adapters, Pfeiffer adapters, knowledge distillation
   (llama teacher -> 2x smaller shared student, and the same student without KD), full fine-tuning as a reference, and one llama run WITHOUT Phase-1 weights (ablation);
   long documents: first 512 tokens / most central sentences / lead + central (decoder-only), 896 tokens (encoder-decoder);
   decoding study: summary length 128 / 256 / 384 / full and repeated-n-gram blocking off / 3 / 4 / 6, fixed vs model-chosen length.
4. Cell 8 - results of every run on validation with the OFFICIAL `evaluate_phase6.py` (BERTScore F1, ROUGE-1, ROUGE-L), plus
   redundancy, abstractiveness, number (legal fact) preservation, memory, speed, the extractive lead baseline, best run per model.
5. Cell 9 - round 2: re-decodes every model's best run with the best decoding setting (only if it helped) and builds consensus
   (MBR) combinations of several runs.

Validation summaries are used only to choose checkpoints and settings - never for training. Test summaries are never empty
(the evaluator rejects the whole file otherwise).

In [1]:
# ===================== P6 CELL 1 - Setup + metric libraries + our tokenizer (Phase 6: Abstractive Summarisation) =====================
# Settings: Accelerator = GPU T4 x2, Internet = On, Secret "HF_TOKEN" (or "HF") switched ON for this notebook.
import os, sys, json, time, math, random, re, shutil, collections, subprocess
os.environ["HF_HOME"] = "/tmp/hf_home"
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge-score==0.1.2", "bert-score==0.3.13"], capture_output=True, text=True)
print("pip:", "ok" if r.returncode == 0 else r.stderr[-1500:])
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi, hf_hub_download
_sec = UserSecretsClient()
try:
    HF_TOKEN = _sec.get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = _sec.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
HF_USER   = api.whoami()["name"]
DATA_REPO = f"{HF_USER}/cs781-p1-data"          # our tokenizer
CKPT_REPO = f"{HF_USER}/cs781-p1-ckpt"          # Phase-1 models; Phase-6 outputs go to its phase6/ folder
SRC_REPO  = "Exploration-Lab/CS-781-Capstone"
WORK      = "/kaggle/working"
P6        = "/tmp/cs781/p6"; os.makedirs(P6, exist_ok=True)
RAW       = f"{P6}/flat"; os.makedirs(RAW, exist_ok=True)
DATA      = f"{P6}/data"; os.makedirs(DATA, exist_ok=True)       # token cache (train/val/test.npz)

# ---- our tokenizer (Phase 1) ----
TOK_PATH = hf_hub_download(DATA_REPO, "tokenizer.json", repo_type="dataset", token=HF_TOKEN, local_dir=f"{P6}/tok")
shutil.copy(TOK_PATH, f"{DATA}/tokenizer.json")
from tokenizers import Tokenizer
tok = Tokenizer.from_file(TOK_PATH)
print("tokenizer vocab:", tok.get_vocab_size())

import torch, rouge_score, bert_score
print(f"torch {torch.__version__} | bert-score {bert_score.__version__} | GPUs {torch.cuda.device_count()}",
      *[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


pip: ok


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


tokenizer.json:   0%|          | 0.00/2.32M [00:00<?, ?B/s]

tokenizer vocab: 32768
torch 2.11.0+cu128 | bert-score 0.3.12 | GPUs 2 Tesla T4 Tesla T4


In [2]:
%%writefile /kaggle/working/model.py
"""CS-781 Phase 1 - four legal language-model architectures, written from scratch in PyTorch.

  gpt2    : classic decoder-only Transformer (learned absolute positions, LayerNorm, GELU MLP, multi-head attention)
  llama   : modern decoder-only Transformer (RoPE, RMSNorm, SwiGLU, grouped-query attention, no biases)
  shared  : lightweight decoder (ALBERT-style factorised embeddings + cycle cross-layer weight sharing + multi-query attn)
  encdec  : encoder-decoder Transformer (bidirectional encoder over the prompt, causal decoder with cross-attention)

All models: tied input/output embeddings, pre-norm residual blocks, PyTorch SDPA attention, KV-cache generation.
"""
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    arch: str = "llama"
    vocab_size: int = 32768
    d_model: int = 768
    n_layer: int = 12            # decoder-only: layers; encdec: layers in EACH of encoder and decoder; shared: effective depth
    n_head: int = 12
    n_kv_head: int = 4           # GQA groups (llama/encdec); shared uses 1 (MQA); gpt2 uses n_head (MHA)
    d_ff: int = 2304
    max_seq_len: int = 1024
    rope_theta: float = 10000.0
    dropout: float = 0.0
    n_unique_layers: int = 6     # shared only: number of distinct blocks, reused cyclically
    d_embed: int = 256           # shared only: factorised embedding size
    norm_eps: float = 1e-5

    def to_dict(self):
        return asdict(self)


def preset(arch: str, **overrides) -> ModelConfig:
    base = {
        "gpt2":   dict(arch="gpt2", d_model=768, n_layer=12, n_head=12, n_kv_head=12, d_ff=3072),
        "llama":  dict(arch="llama", d_model=768, n_layer=12, n_head=12, n_kv_head=4, d_ff=2304),
        "shared": dict(arch="shared", d_model=768, n_layer=12, n_head=12, n_kv_head=1, d_ff=3072,
                       n_unique_layers=6, d_embed=256),
        "encdec": dict(arch="encdec", d_model=768, n_layer=6, n_head=12, n_kv_head=4, d_ff=2304),
    }[arch]
    base.update(overrides)
    return ModelConfig(**base)


# ----------------------------------------------------------------------------------------------------------------
# building blocks
# ----------------------------------------------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).to(x.dtype)


class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.ln = nn.LayerNorm(d, eps=eps)

    def forward(self, x):
        return F.layer_norm(x.float(), self.ln.normalized_shape, self.ln.weight.float(), self.ln.bias.float(),
                            self.ln.eps).to(x.dtype)


def rope_cache(head_dim, max_len, theta, device=None):
    inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_len, device=device).float()
    freqs = torch.outer(t, inv)                       # (T, hd/2)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    # x: (B, H, T, hd); cos/sin: (T, hd/2) - rotate pairs (x[..., :hd/2], x[..., hd/2:])
    xf = x.float()
    x1, x2 = xf[..., : xf.shape[-1] // 2], xf[..., xf.shape[-1] // 2:]
    cos, sin = cos[None, None], sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.to(x.dtype)


class Attention(nn.Module):
    """Multi-head / grouped-query / multi-query attention. Self-attention (optionally causal) or cross-attention."""

    def __init__(self, cfg: ModelConfig, n_kv_head: int, bias: bool, use_rope: bool, cross: bool = False):
        super().__init__()
        d, h = cfg.d_model, cfg.n_head
        assert d % h == 0 and h % n_kv_head == 0
        self.h, self.kv, self.hd = h, n_kv_head, d // h
        self.use_rope, self.cross = use_rope, cross
        self.q = nn.Linear(d, h * self.hd, bias=bias)
        self.k = nn.Linear(d, n_kv_head * self.hd, bias=bias)
        self.v = nn.Linear(d, n_kv_head * self.hd, bias=bias)
        self.o = nn.Linear(h * self.hd, d, bias=bias)
        self.o.is_residual_proj = True
        self.dropout = cfg.dropout

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.hd).transpose(1, 2)          # (B, n, T, hd)

    def _expand(self, x):
        return x if self.kv == self.h else x.repeat_interleave(self.h // self.kv, dim=1)

    def forward(self, x, rope=None, pos=0, causal=True, cache=None, memory=None):
        """
        x: (B, T, d).  rope: (cos, sin) full tables.  pos: absolute position of x[:,0] (for RoPE during decoding).
        cache: dict used during generation. Self-attn: keys 'k','v' accumulate. Cross-attn: 'k','v' computed once.
        memory: encoder output (B, S, d) for cross-attention.
        """
        q = self._split(self.q(x), self.h)
        if self.cross:
            if cache is not None and "k" in cache:
                k, v = cache["k"], cache["v"]
            else:
                k, v = self._split(self.k(memory), self.kv), self._split(self.v(memory), self.kv)
                if cache is not None:
                    cache["k"], cache["v"] = k, v
            is_causal = False
        else:
            k, v = self._split(self.k(x), self.kv), self._split(self.v(x), self.kv)
            if self.use_rope:
                cos, sin = rope
                T = x.shape[1]
                q = apply_rope(q, cos[pos:pos + T], sin[pos:pos + T])
                k = apply_rope(k, cos[pos:pos + T], sin[pos:pos + T])
            if cache is not None:
                if "k" in cache:
                    k = torch.cat([cache["k"], k], dim=2); v = torch.cat([cache["v"], v], dim=2)
                cache["k"], cache["v"] = k, v
            # causal mask only needed when queries and keys cover the same span (training / prefill)
            is_causal = causal and q.shape[2] == k.shape[2] and q.shape[2] > 1
            if causal and q.shape[2] != k.shape[2] and q.shape[2] != 1:
                raise ValueError("multi-token decode with a non-empty cache is not supported")
        y = F.scaled_dot_product_attention(q, self._expand(k), self._expand(v), is_causal=is_causal,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(x.shape[0], x.shape[1], self.h * self.hd)
        return self.o(y)


class MLP(nn.Module):
    def __init__(self, cfg: ModelConfig, kind: str, bias: bool):
        super().__init__()
        self.kind = kind
        if kind == "swiglu":
            self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
            self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
        else:
            self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=bias)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=bias)
        self.down.is_residual_proj = True
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        if self.kind == "swiglu":
            h = F.silu(self.gate(x)) * self.up(x)
        else:
            h = F.gelu(self.up(x), approximate="tanh")
        return self.drop(self.down(h))


class Block(nn.Module):
    """Pre-norm Transformer block: x + Attn(Norm(x)) [+ CrossAttn(Norm(x), memory)] + MLP(Norm(x))."""

    def __init__(self, cfg: ModelConfig, style: str, n_kv_head: int, cross: bool = False):
        super().__init__()
        gpt2 = style == "gpt2"
        Norm = LayerNorm if gpt2 else RMSNorm
        self.n1 = Norm(cfg.d_model, cfg.norm_eps)
        self.attn = Attention(cfg, n_kv_head, bias=gpt2, use_rope=not gpt2)
        self.has_cross = cross
        if cross:
            self.nc = Norm(cfg.d_model, cfg.norm_eps)
            self.xattn = Attention(cfg, n_kv_head, bias=False, use_rope=False, cross=True)
        self.n2 = Norm(cfg.d_model, cfg.norm_eps)
        self.mlp = MLP(cfg, "gelu" if gpt2 else "swiglu", bias=gpt2)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope=None, pos=0, causal=True, cache=None, memory=None, xcache=None):
        x = x + self.drop(self.attn(self.n1(x), rope=rope, pos=pos, causal=causal, cache=cache))
        if self.has_cross:
            x = x + self.drop(self.xattn(self.nc(x), memory=memory, cache=xcache))
        return x + self.mlp(self.n2(x))


# ----------------------------------------------------------------------------------------------------------------
# the language model
# ----------------------------------------------------------------------------------------------------------------
class LegalLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        a = cfg.arch
        assert a in ("gpt2", "llama", "shared", "encdec"), a
        d = cfg.d_model
        if a == "shared":
            self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_embed)
            self.emb_proj = nn.Linear(cfg.d_embed, d, bias=False)          # E -> d   (factorised embedding)
        else:
            self.tok_emb = nn.Embedding(cfg.vocab_size, d)
        if a == "gpt2":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, d)
        self.drop = nn.Dropout(cfg.dropout)

        if a == "gpt2":
            self.blocks = nn.ModuleList([Block(cfg, "gpt2", cfg.n_head) for _ in range(cfg.n_layer)])
        elif a == "llama":
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_layer)])
        elif a == "shared":
            assert cfg.n_layer % cfg.n_unique_layers == 0
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_unique_layers)])
        else:  # encdec
            self.enc_blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head) for _ in range(cfg.n_layer)])
            self.enc_norm = RMSNorm(d, cfg.norm_eps)
            self.blocks = nn.ModuleList([Block(cfg, "llama", cfg.n_kv_head, cross=True) for _ in range(cfg.n_layer)])
        self.norm = LayerNorm(d, cfg.norm_eps) if a == "gpt2" else RMSNorm(d, cfg.norm_eps)

        if a != "gpt2":
            hd = d // cfg.n_head
            cos, sin = rope_cache(hd, max(2 * cfg.max_seq_len, 2048), cfg.rope_theta)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)
        n_res = 2 * self.n_layer_applications() + (cfg.n_layer if a == "encdec" else 0)
        for name, p in self.named_parameters():
            if name.endswith("o.weight") or name.endswith("down.weight"):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(n_res))

    # -- helpers ------------------------------------------------------------------------------------------------
    def n_layer_applications(self):
        return self.cfg.n_layer * (2 if self.cfg.arch == "encdec" else 1)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def _rope(self):
        return None if self.cfg.arch == "gpt2" else (self.rope_cos, self.rope_sin)

    def _embed(self, idx, pos=0):
        x = self.tok_emb(idx)
        if self.cfg.arch == "shared":
            x = self.emb_proj(x)
        if self.cfg.arch == "gpt2":
            T = idx.shape[1]
            if pos + T > self.cfg.max_seq_len:
                raise ValueError(f"gpt2 positions exceed max_seq_len ({pos + T} > {self.cfg.max_seq_len})")
            x = x + self.pos_emb(torch.arange(pos, pos + T, device=idx.device))[None]
        return self.drop(x)

    def _logits(self, h):
        h = self.norm(h)
        if self.cfg.arch == "shared":
            h = F.linear(h, self.emb_proj.weight.t())     # d -> E   (tied with the input projection)
        return F.linear(h, self.tok_emb.weight)            # tied output embedding

    def _decoder_layers(self):
        if self.cfg.arch == "shared":                       # cycle sharing: 0,1,..,5,0,1,..,5
            return [self.blocks[i % self.cfg.n_unique_layers] for i in range(self.cfg.n_layer)]
        return list(self.blocks)

    def encode(self, src):
        x = self._embed(src)
        for blk in self.enc_blocks:
            x = blk(x, rope=self._rope(), pos=0, causal=False)
        return self.enc_norm(x)

    # -- training forward -----------------------------------------------------------------------------------------
    def forward(self, idx, targets=None, src=None, loss_mask=None):
        """Decoder-only: idx (B,T) -> logits; loss vs targets (B,T).  encdec: src (B,S) is the encoder input."""
        memory = self.encode(src) if self.cfg.arch == "encdec" else None
        x = self._embed(idx)
        for blk in self._decoder_layers():
            x = blk(x, rope=self._rope(), pos=0, causal=True, memory=memory)
        logits = self._logits(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), reduction="none")
        if loss_mask is not None:
            m = loss_mask.reshape(-1).float()
            return logits, (loss * m).sum() / m.sum().clamp(min=1)
        return logits, loss.mean()

    # -- generation -----------------------------------------------------------------------------------------------
    @torch.no_grad()
    def generate(self, prompt, max_new_tokens, temperature=1.0, top_k=0, top_p=1.0, repetition_penalty=1.0,
                 no_repeat_ngram=0, banned_ids=(), bos_id=2, generator=None, use_cache=True):
        """prompt: (B, T) LongTensor, all rows the same length (no padding). Returns (B, max_new_tokens) new ids.
        temperature 0 = greedy."""
        B = prompt.shape[0]
        dev = prompt.device
        layers = self._decoder_layers()
        if self.cfg.arch == "encdec":
            memory = self.encode(prompt)
            seq = torch.full((B, 1), bos_id, dtype=torch.long, device=dev)   # decoder starts from <|bos|>
            history = prompt                                                   # for repetition penalties
        else:
            memory, seq, history = None, prompt, prompt
        caches = [dict() for _ in layers] if use_cache else None
        xcaches = [dict() for _ in layers] if (use_cache and memory is not None) else [None] * len(layers)
        out = []
        cur, pos = seq, 0
        for step in range(max_new_tokens):
            if use_cache:
                x = self._embed(cur, pos=pos)
                for li, blk in enumerate(layers):
                    x = blk(x, rope=self._rope(), pos=pos, causal=True, cache=caches[li], memory=memory,
                            xcache=xcaches[li])
                pos += cur.shape[1]
            else:
                x = self._embed(seq)
                for blk in layers:
                    x = blk(x, rope=self._rope(), pos=0, causal=True, memory=memory)
            logits = self._logits(x[:, -1:, :])[:, -1, :].float()
            full_hist = torch.cat([history] + out, dim=1) if out else history
            nxt = sample_next(logits, full_hist, temperature, top_k, top_p, repetition_penalty, no_repeat_ngram,
                              banned_ids, generator)
            out.append(nxt)
            seq = torch.cat([seq, nxt], dim=1)
            cur = nxt
        return torch.cat(out, dim=1)


def sample_next(logits, history, temperature, top_k, top_p, repetition_penalty, no_repeat_ngram, banned_ids,
                generator=None):
    """logits (B,V) float32, history (B,L) long (prompt + generated so far). Returns (B,1)."""
    logits = logits.clone()
    if banned_ids:
        logits[:, list(banned_ids)] = -float("inf")
    if repetition_penalty != 1.0:
        # CTRL-style penalty on tokens already present in the recent history (last 256 tokens)
        recent = history[:, -256:]
        sc = torch.gather(logits, 1, recent)
        sc = torch.where(sc > 0, sc / repetition_penalty, sc * repetition_penalty)
        logits.scatter_(1, recent, sc)
    if no_repeat_ngram and history.shape[1] >= no_repeat_ngram:
        n = no_repeat_ngram
        hist = history[:, -512:].tolist()
        for b, h in enumerate(hist):
            prefix = tuple(h[-(n - 1):]) if n > 1 else tuple()
            banned = {tuple(h[i:i + n])[-1] for i in range(len(h) - n + 1) if tuple(h[i:i + n - 1]) == prefix}
            if banned:
                logits[b, list(banned)] = -float("inf")
    if temperature <= 0:
        return logits.argmax(dim=-1, keepdim=True)
    logits = logits / temperature
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(top_k, logits.shape[-1]), dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        cp = torch.softmax(sl, dim=-1).cumsum(dim=-1)
        remove = cp - torch.softmax(sl, dim=-1) > top_p          # keep the smallest set with mass >= top_p
        sl = sl.masked_fill(remove, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, si, sl)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1, generator=generator)


Writing /kaggle/working/model.py


In [3]:
%%writefile /kaggle/working/p6data.py
"""CS-781 Phase 6 (abstractive summarisation): read train/test once, normalise like our Phase-1 text, split documents into
sentences, tokenise with OUR tokenizer, score every sentence's importance (TF-IDF similarity to the whole document, computed
from the document itself), hold out a validation split, and cache everything -> <out_dir>/{train,val,test}.npz."""
import argparse, json, os, random, re, time, unicodedata
import numpy as np

PUNCT = {"‘": "'", "’": "'", "‚": "'", "‛": "'", "′": "'",
         "“": '"', "”": '"', "„": '"', "″": '"',
         "‐": "-", "‑": "-", "‒": "-", "–": "-", "—": "-",
         "―": "-", "−": "-", "•": "*", "·": "."}
_TABLE = str.maketrans(PUNCT)
_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
_WS = re.compile(r"\s+")
_SENT = re.compile(r"(?<=[.?!;:])\s+(?=[A-Z0-9(\"'\[])")


def normalize(t):
    """Same cleaning as Phase 1-5 (fancy punctuation -> ASCII, NFKC, accents stripped, non-ASCII / control chars dropped,
    whitespace collapsed)."""
    if not isinstance(t, str):
        t = " ".join(t) if isinstance(t, list) else str(t or "")
    t = t.translate(_TABLE)
    if not t.isascii():
        t = unicodedata.normalize("NFKC", t)
        t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode("ascii")
    t = _CTRL.sub("", t)
    return _WS.sub(" ", t).strip()


def load_records(path):
    d = json.load(open(path, encoding="utf-8"))
    if isinstance(d, dict):
        if "data" in d and isinstance(d["data"], list):
            d = d["data"]
        else:
            d = [dict(v, id=k) if isinstance(v, dict) else {"id": k, "document": v} for k, v in d.items()]
    out = []
    for r in d:
        doc = r.get("document", r.get("text", r.get("judgment", "")))
        summ = r.get("summary", r.get("reference", r.get("abstract")))
        out.append((str(r.get("id", r.get("doc_id", r.get("name")))), doc, summ))
    return out


def importance(sents):
    """TF-IDF cosine of every sentence with the whole document (centrality); no outside model."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    if len(sents) < 2:
        return np.ones(len(sents), np.float32)
    try:
        X = TfidfVectorizer(stop_words="english", sublinear_tf=True, min_df=1).fit_transform(sents)
    except ValueError:                                   # only stop words
        return np.ones(len(sents), np.float32)
    c = np.asarray(X.mean(0)).ravel()
    c = c / max(np.linalg.norm(c), 1e-12)
    return np.asarray(X @ c).ravel().astype(np.float32)


def pack(recs, tok, labelled):
    doc_ids, doc_off, s_off, s_ptr, s_score, sum_ids, sum_off, names, nsw = [], [0], [], [0], [], [], [0], [], []
    for did, doc, summ in recs:
        sents = [s for s in _SENT.split(normalize(doc)) if s.strip()] or [""]
        encs = tok.encode_batch([" " + s for s in sents], add_special_tokens=False)
        pos = doc_off[-1]
        for e in encs:
            s_off.append(pos - doc_off[-1]); doc_ids.extend(e.ids); pos += len(e.ids)
        s_score.extend(importance(sents).tolist())
        doc_off.append(pos); s_ptr.append(len(s_off)); names.append(did)
        if labelled:
            st = normalize(summ)
            ids = tok.encode(st, add_special_tokens=False).ids
            sum_ids.extend(ids); nsw.append(len(st.split()))
        sum_off.append(len(sum_ids))
    return dict(doc=np.asarray(doc_ids, np.uint16), doc_off=np.asarray(doc_off, np.int64), s_off=np.asarray(s_off, np.int64),
                s_ptr=np.asarray(s_ptr, np.int64), s_score=np.asarray(s_score, np.float16), sum=np.asarray(sum_ids, np.uint16),
                sum_off=np.asarray(sum_off, np.int64), name=np.asarray(names)), nsw


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--raw_dir", required=True)
    p.add_argument("--tok", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--n_val", type=int, default=200)
    p.add_argument("--seed", type=int, default=6)
    a = p.parse_args(argv)
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(a.tok)
    os.makedirs(a.out_dir, exist_ok=True)
    t0 = time.time()
    tr = load_records(os.path.join(a.raw_dir, "train.json"))
    te = load_records(os.path.join(a.raw_dir, "test.json"))
    seen, dup = set(), 0
    trk = []
    for r in tr:
        if r[0] in seen or r[2] is None or not str(r[2]).strip():
            dup += 1; continue
        seen.add(r[0]); trk.append(r)
    te_seen, tek = set(), []
    for r in te:
        if r[0] not in te_seen:
            te_seen.add(r[0]); tek.append(r)
    rng = random.Random(a.seed)
    idx = list(range(len(trk))); rng.shuffle(idx)
    val_set = set(idx[:a.n_val])
    val = [trk[i] for i in sorted(val_set)]; train = [trk[i] for i in range(len(trk)) if i not in val_set]
    stats = {"train_records": len(tr), "train_kept": len(trk), "dropped_duplicate_or_empty": dup, "n_train": len(train),
             "n_val": len(val), "n_test": len(tek), "test_records": len(te)}
    for name, recs, lab in [("train", train, True), ("val", val, True), ("test", tek, False)]:
        arrs, nsw = pack(recs, tok, lab)
        np.savez(os.path.join(a.out_dir, f"{name}.npz"), **arrs)
        dl = np.diff(arrs["doc_off"]); sl = np.diff(arrs["sum_off"])
        stats[name] = {"docs": len(recs), "doc_tokens_mean": float(dl.mean()), "doc_tokens_median": float(np.median(dl)),
                       "doc_tokens_p90": float(np.percentile(dl, 90)), "sentences_per_doc": float(np.diff(arrs["s_ptr"]).mean())}
        if lab:
            stats[name].update({"summary_tokens_mean": float(sl.mean()), "summary_tokens_median": float(np.median(sl)),
                                "summary_tokens_p10": float(np.percentile(sl, 10)), "summary_tokens_p90": float(np.percentile(sl, 90)),
                                "summary_words_mean": float(np.mean(nsw)), "summary_words_median": float(np.median(nsw)),
                                "frac_summary_over_512_tokens": float((sl > 512).mean())})
        print(name, json.dumps(stats[name]), flush=True)
    json.dump({r[0]: str(r[2]) for r in val}, open(os.path.join(a.out_dir, "val_truth.json"), "w"))   # official truth format
    sp = {}
    for c in ["<|endoftext|>", "<|eos|>", "</s>", "<eos>", "<|bos|>", "<s>", "<bos>", "<|pad|>", "<pad>", "<|sep|>", "<sep>"]:
        i = tok.token_to_id(c)
        if i is not None:
            sp[c] = int(i)
    stats["special_tokens"] = sp
    stats["seconds"] = round(time.time() - t0, 1)
    print("special tokens in our tokenizer:", sp, "| splits:", {k: stats[k] for k in ["n_train", "n_val", "n_test", "dropped_duplicate_or_empty"]}, flush=True)
    json.dump(stats, open(os.path.join(a.out_dir, "stats.json"), "w"), indent=1)


if __name__ == "__main__":
    main()


Writing /kaggle/working/p6data.py


In [4]:
%%writefile /kaggle/working/p6sum.py
"""CS-781 Phase 6: abstractive legal summarisation with our own Phase-1 models.

decoder-only (llama / gpt2 / shared): [selected document tokens (<= P)] [SEP] [summary] [EOS], loss on the summary only;
                                      at test time the model GENERATES the summary after [SEP] (KV-cache decoding).
encoder-decoder (encdec)            : encoder reads the selected document (<= enc_len tokens), decoder generates the summary.
Long documents ("select"): lead (first P tokens) | central (most important sentences, TF-IDF centrality, in original order)
                           | leadcentral (first P/2 tokens + most important later sentences).
Methods: lora | lorma (compulsory) | houlsby (adapters after attention + feed-forward) | pfeiffer (adapter after feed-forward)
         | full ; --kd_teacher adds word-level knowledge distillation (student matches the teacher's next-token distribution).
Model selection on a held-out validation split (never trained on): validation loss during training, then the OFFICIAL
evaluate_phase6.py (BERTScore F1, ROUGE-1, ROUGE-L) on generated validation summaries.
"""
import argparse, json, math, os, random, re, subprocess, sys, time, zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from model import LegalLM, ModelConfig, sample_next          # our Phase-1 model code (unchanged)


# ================================================================================================================
# adaptation modules (our own code)
# ================================================================================================================
class LoRALinear(nn.Module):
    """LoRA (Hu et al. 2021): y = W0 x + (alpha/r) B A x ; B = 0 at start."""

    def __init__(self, base: nn.Linear, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)


class LoRMALinear(nn.Module):
    """LoRMA (Bihany, Patel & Modi, 2025), additive rank inflation: W' = (I + (alpha/r) B A) W0 ; B = 0 at start."""

    def __init__(self, base: nn.Linear, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base, self.scale = base, alpha / r
        d_out = base.out_features
        self.A = nn.Parameter(torch.randn(r, d_out) / math.sqrt(d_out))
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        y = y0 + self.scale * F.linear(F.linear(self.drop(y0), self.A), self.B)
        return y if self.base.bias is None else y + self.base.bias


class AdapterOut(nn.Module):
    """Bottleneck adapter on a sub-layer output: y = W x ; y + Up(GELU(Down(y))) ; Up = 0 at start.
    Houlsby et al. (2019): after BOTH the attention and the feed-forward sub-layer.  Pfeiffer et al. (2020): after the
    feed-forward sub-layer only."""

    def __init__(self, base: nn.Linear, dim=64):
        super().__init__()
        self.base = base
        d = base.out_features
        self.a_down = nn.Linear(d, dim)
        self.a_up = nn.Linear(dim, d)
        nn.init.zeros_(self.a_up.weight); nn.init.zeros_(self.a_up.bias)

    def forward(self, x):
        y = self.base(x)
        return y + self.a_up(F.gelu(self.a_down(y)))


def roots(lm):
    return [lm.enc_blocks, lm.blocks] if lm.cfg.arch == "encdec" else [lm.blocks]


def setup_method(lm, a):
    for p in lm.parameters():
        p.requires_grad = True
    m = a["method"]
    if m == "full":
        return
    for p in lm.parameters():
        p.requires_grad = False
    n = 0
    for root in roots(lm):
        for mod in list(root.modules()):
            for name, ch in list(mod.named_children()):
                if not isinstance(ch, nn.Linear):
                    continue
                if m in ("lora", "lorma") and name in a["targets"].split(","):
                    setattr(mod, name, (LoRMALinear if m == "lorma" else LoRALinear)(ch, a["rank"], a["alpha"], 0.05)); n += 1
                elif (m == "houlsby" and name in ("o", "down")) or (m == "pfeiffer" and name == "down"):
                    setattr(mod, name, AdapterOut(ch, a["adapter_dim"])); n += 1
    print(f"{m}: wrapped {n} layers", flush=True)


def count_params(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)


# ================================================================================================================
# data
# ================================================================================================================
class Split:
    def __init__(self, path, limit=0):
        z = np.load(path, allow_pickle=False)
        for k in ["doc", "doc_off", "s_off", "s_ptr", "s_score", "sum", "sum_off"]:
            setattr(self, k, z[k])
        self.name = [str(x) for x in z["name"]]
        self.n = len(self.name) if not limit else min(limit, len(self.name))

    def document(self, i):
        return self.doc[self.doc_off[i]:self.doc_off[i + 1]]

    def summary(self, i):
        return self.sum[self.sum_off[i]:self.sum_off[i + 1]]

    def select(self, i, mode, P):
        """-> np array of min(len(document), P) document tokens.
        lead        : the first P tokens
        central     : the most central sentences (TF-IDF similarity to the whole document), kept in document order
        leadcentral : the first P/2 tokens + the most central LATER sentences
        The last chosen sentence is cut so that every reading has exactly P tokens (same-length batches, no padding)."""
        d = self.document(i); n = len(d)
        if n == 0:                                             # empty document: one end-of-text token (never an empty input)
            return np.zeros(1, d.dtype)
        if mode == "lead" or n <= P:
            return d[:P]
        st = self.s_off[self.s_ptr[i]:self.s_ptr[i + 1]]; en = np.append(st[1:], n)
        sc = self.s_score[self.s_ptr[i]:self.s_ptr[i + 1]].astype(np.float32)
        head = P // 2 if mode == "leadcentral" else 0
        budget, picked, partial = P - head, [], None
        for j in np.argsort(-sc, kind="stable"):
            if st[j] < head:                                   # already inside the lead part
                continue
            ln = int(en[j] - st[j])
            if ln <= budget:
                picked.append((int(st[j]), ln)); budget -= ln
            elif partial is None:
                partial = int(st[j])
            if budget == 0:
                break
        if budget > 0 and partial is not None:
            picked.append((partial, budget)); budget = 0
        parts = [d[:head]] + [d[a:a + ln] for a, ln in sorted(picked)]
        out = np.concatenate(parts)
        if len(out) < P:                                       # (rare) top up with the following lead tokens
            out = np.concatenate([out, d[head:head + P - len(out)]])
        return out[:P]


# ================================================================================================================
# forward passes
# ================================================================================================================
def hidden(lm, inp, src=None):
    if lm.cfg.arch == "encdec":
        memory = lm.encode(src)
        x = lm._embed(inp)
        for blk in lm._decoder_layers():
            x = blk(x, rope=lm._rope(), pos=0, causal=True, memory=memory)
        return x
    x = lm._embed(inp)
    for blk in lm._decoder_layers():
        x = blk(x, rope=lm._rope(), pos=0, causal=True)
    return x


def make_batch(items, arch, pad, device):
    """items: list of dicts (inp, tgt, mask[, src]). Right-padding is safe for the causal decoder (masked in the loss);
    encoder inputs in one batch always have the same length (batches are grouped by encoder length)."""
    T = max(len(it["inp"]) for it in items)
    inp = torch.full((len(items), T), pad, dtype=torch.long); tgt = torch.full_like(inp, pad); m = torch.zeros(len(items), T, dtype=torch.bool)
    for b, it in enumerate(items):
        L = len(it["inp"]); inp[b, :L] = torch.as_tensor(it["inp"]); tgt[b, :L] = torch.as_tensor(it["tgt"]); m[b, :L] = torch.as_tensor(it["mask"])
    src = None
    if arch == "encdec":
        assert len(set(len(it["src"]) for it in items)) == 1
        src = torch.as_tensor(np.stack([it["src"] for it in items]).astype(np.int64)).to(device)
    return inp.to(device), tgt.to(device), m.to(device), src


def sel_logits(lm, inp, m, src):
    h = hidden(lm, inp, src)
    return lm._logits(h[m]).float()                                      # (N_summary_tokens, V)


# ================================================================================================================
# generation (KV cache; same-length prompts in a batch; no-repeat n-grams counted on the generated text only)
# ================================================================================================================
def block_repeats(logits, gen, n):
    """In-place: forbid any token that would repeat an n-gram already present in the generated summary (vectorised)."""
    B, t = gen.shape
    if n <= 0 or t < n - 1 or t == 0:
        return
    if n == 1:
        logits.scatter_(1, gen, -float("inf")); return
    if t < n:
        return
    win = gen.unfold(1, n - 1, 1)                                  # (B, t-n+2, n-1)
    match = (win[:, :-1] == gen[:, t - n + 1:][:, None, :]).all(-1)  # windows equal to the current suffix
    pen = torch.zeros_like(logits).scatter_add_(1, gen[:, n - 1:], match.float())
    logits.masked_fill_(pen > 0, -float("inf"))


@torch.no_grad()
def generate(lm, prompt, max_new, min_new, eos, bos, no_repeat=4, rep_pen=1.0, temperature=0.0, top_p=1.0):
    """prompt (B,T) same-length rows. decoder-only: continue after the prompt; encdec: prompt = encoder input,
    the decoder starts from <|bos|>. Greedy by default. Returns (B, <=max_new) numpy ids."""
    B, dev = prompt.shape[0], prompt.device
    layers = lm._decoder_layers()
    if lm.cfg.arch == "encdec":
        memory = lm.encode(prompt); cur = torch.full((B, 1), bos, dtype=torch.long, device=dev)
    else:
        memory, cur = None, prompt
    caches = [dict() for _ in layers]
    xcaches = [dict() for _ in layers] if memory is not None else [None] * len(layers)
    gen = torch.zeros((B, 0), dtype=torch.long, device=dev)
    pos, done = 0, torch.zeros(B, dtype=torch.bool, device=dev)
    for step in range(max_new):
        x = lm._embed(cur, pos=pos)
        for li, blk in enumerate(layers):
            x = blk(x, rope=lm._rope(), pos=pos, causal=True, cache=caches[li], memory=memory, xcache=xcaches[li])
        pos += cur.shape[1]
        logits = lm._logits(x[:, -1, :]).float()
        if step < min_new:
            logits[:, eos] = -float("inf")
        block_repeats(logits, gen, no_repeat)
        nxt = sample_next(logits, gen if gen.shape[1] else prompt[:, :1], temperature, 0, top_p, rep_pen if gen.shape[1] else 1.0, 0, (), None)
        nxt[done] = eos
        gen = torch.cat([gen, nxt], 1)
        done |= nxt[:, 0] == eos
        if step % 32 == 31 and bool(done.all()):
            break
        cur = nxt
    return gen.cpu().numpy()


# ================================================================================================================
SEP_TEXT = " Summary:"          # plain-text separator between the document and the summary (decoder-only models)
_NUM = re.compile(r"\b\d[\d/.,-]*\d\b|\b\d\b")


def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--run_name", required=True)
    p.add_argument("--base_run", required=True)
    p.add_argument("--base_file", default="best_model.pt")
    p.add_argument("--method", default="lorma", choices=["lora", "lorma", "houlsby", "pfeiffer", "full"])
    p.add_argument("--kd_teacher", default="", help="Phase-6 run used as teacher (its best.pt)")
    p.add_argument("--kd_alpha", type=float, default=0.5, help="weight of the gold-summary loss (1-alpha: teacher)")
    p.add_argument("--kd_T", type=float, default=2.0)
    p.add_argument("--select", default="lead", choices=["lead", "central", "leadcentral"])
    p.add_argument("--prompt_len", type=int, default=512, help="document tokens for decoder-only models")
    p.add_argument("--enc_len", type=int, default=896, help="document tokens for the encoder-decoder (Phase-1 encoder saw up to 896)")
    p.add_argument("--dec_len", type=int, default=640, help="summary tokens for the encoder-decoder")
    p.add_argument("--bs", type=int, default=8)
    p.add_argument("--epochs", type=float, default=6)
    p.add_argument("--max_train_min", type=float, default=30)
    p.add_argument("--lr", type=float, default=5e-5, help="full fine-tuning LR")
    p.add_argument("--peft_lr", type=float, default=5e-4)
    p.add_argument("--rank", type=int, default=16)
    p.add_argument("--alpha", type=float, default=32)
    p.add_argument("--targets", default="q,k,v,o,gate,up,down", help="LoRA / LoRMA: linear layers to adapt")
    p.add_argument("--adapter_dim", type=int, default=64)
    p.add_argument("--wd", type=float, default=0.01)
    p.add_argument("--warmup", type=float, default=0.05)
    p.add_argument("--seed", type=int, default=13)
    # decoding (generation length + decoding strategy)
    p.add_argument("--gen_max", type=int, default=0, help="max new tokens (0 = as long as the context allows)")
    p.add_argument("--gen_min_frac", type=float, default=0.8, help="no end-of-summary before this share of gen_max")
    p.add_argument("--no_repeat", type=int, default=4, help="block repeated n-grams (in tokens) inside the summary; 0 = off")
    p.add_argument("--rep_pen", type=float, default=1.0)
    p.add_argument("--temperature", type=float, default=0.0)
    p.add_argument("--top_p", type=float, default=1.0)
    p.add_argument("--gen_bs", type=int, default=32)
    p.add_argument("--from_scratch", type=int, default=0, help="1 = ablation: same architecture, random weights (no Phase-1 pretraining)")
    p.add_argument("--decode_only", type=int, default=0, help="1 = no training: load --init_run and only generate")
    p.add_argument("--init_run", default="", help="start from this Phase-6 run's weights (continue training / re-decode)")
    p.add_argument("--data_dir", required=True)
    p.add_argument("--ckpt_dir", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--hf_repo", default="")
    p.add_argument("--max_steps", type=int, default=-1)
    p.add_argument("--train_limit", type=int, default=0)
    p.add_argument("--val_limit", type=int, default=0)
    p.add_argument("--test_limit", type=int, default=0)
    return p.parse_args(argv)


def hf_up(repo, local, remote):
    if not repo:
        return
    from huggingface_hub import HfApi
    for k in range(3):
        try:
            HfApi(token=os.environ.get("HF_TOKEN")).upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=repo, repo_type="model")
            return
        except Exception as e:
            print("[hf] upload failed:", e, flush=True); time.sleep(15 * (k + 1))


def hf_get(repo, remote, local_dir):
    if not repo:
        return None
    from huggingface_hub import hf_hub_download
    try:
        return hf_hub_download(repo, remote, repo_type="model", local_dir=local_dir, token=os.environ.get("HF_TOKEN"))
    except Exception:
        return None


def load_lm(args, base_run, base_file, method_args):
    ck_path = os.path.join(args.ckpt_dir, "runs", base_run, base_file)
    if not os.path.exists(ck_path):
        hf_get(args.hf_repo, f"runs/{base_run}/{base_file}", args.ckpt_dir)
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    cfg = ck["config"]
    lm = LegalLM(ModelConfig(**cfg))
    if getattr(args, "from_scratch", 0) and not method_args.get("_teacher"):
        print("FROM SCRATCH: Phase-1 weights NOT loaded (random initialisation, same architecture) - ablation only", flush=True)
    else:
        missing, _ = lm.load_state_dict(ck["model"], strict=False)
        assert not missing, f"missing backbone weights: {missing[:5]}"
    setup_method(lm, method_args)
    return lm, cfg


def run_file(args, run):
    p = os.path.join(args.out_dir, run, "best.pt")
    if not os.path.exists(p):
        p = hf_get(args.hf_repo, f"phase6/runs/{run}/best.pt", os.path.join(args.out_dir, "_hub", run))
    assert p and os.path.exists(p), f"{run}: best.pt not found (locally or on Hugging Face)"
    return torch.load(p, map_location="cpu", weights_only=False)


def load_run_state(args, run, lm):
    """best.pt holds only the TRAINED weights (adapters for PEFT, everything for full fine-tuning)."""
    ck = run_file(args, run)
    cur = lm.state_dict()
    extra = set(ck["state_dict"]) - set(cur)
    need = set(n for n, p in lm.named_parameters() if p.requires_grad)
    assert not extra and need <= set(ck["state_dict"]), f"{run}: weights do not match (extra {list(extra)[:3]}, missing {list(need - set(ck['state_dict']))[:3]})"
    cur.update({k: v.to(cur[k].dtype) for k, v in ck["state_dict"].items()})
    lm.load_state_dict(cur)
    return ck["args"]


def ngram_stats(gen_tokens, src_tokens, n=3):
    g = [tuple(gen_tokens[i:i + n]) for i in range(len(gen_tokens) - n + 1)]
    if not g:
        return 0.0, 0.0
    s = set(tuple(src_tokens[i:i + n]) for i in range(len(src_tokens) - n + 1))
    rep = 1 - len(set(g)) / len(g)                                  # share of repeated n-grams (redundancy)
    novel = sum(x not in s for x in g) / len(g)                     # share of n-grams not in the input (abstractiveness)
    return rep, novel


def fact_stats(pred, ref, src):
    """legal-fact preservation proxies on numbers (sections, years, dates, amounts):
    recall = share of the reference's numbers that the summary contains; support = share of the summary's numbers
    that occur in the source document (1 - support ~ invented numbers)."""
    pn, rn, sn = set(_NUM.findall(pred)), set(_NUM.findall(ref)), set(_NUM.findall(src))
    return (len(pn & rn) / len(rn) if rn else None), (len(pn & sn) / len(pn) if pn else None)


def official_eval(work, truth, pred):
    r = subprocess.run([sys.executable, os.path.join(work, "evaluate_phase6.py"), "--truth", truth, "--prediction", pred],
                       capture_output=True, text=True)
    txt = r.stdout + r.stderr
    out = {}
    for key, pat in [("bertscore_f1", r"BERTScore F1:\s*([0-9.]+)"), ("rouge1", r"ROUGE-1:\s*([0-9.]+)"), ("rougeL", r"ROUGE-L:\s*([0-9.]+)")]:
        mm = re.search(pat, txt)
        out[key] = float(mm.group(1)) if mm else None
    out["evaluator_output"] = " | ".join(l.strip() for l in txt.strip().splitlines()[-6:])
    return out


def main(argv=None):
    args = parse_args(argv)
    t_start = time.time()
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cuda = dev.type == "cuda"
    amp = (lambda: torch.autocast("cuda", dtype=torch.float16)) if cuda else (lambda: torch.autocast("cpu", enabled=False))
    run_dir = os.path.join(args.out_dir, args.run_name); os.makedirs(run_dir, exist_ok=True)
    logf = open(os.path.join(run_dir, "log.txt"), "a")
    work = os.path.dirname(os.path.abspath(__file__))

    def log(*m):
        s = f"[{time.strftime('%H:%M:%S')}][{args.run_name}] " + " ".join(str(x) for x in m)
        print(s, flush=True); logf.write(s + "\n"); logf.flush()

    from tokenizers import Tokenizer
    stats = json.load(open(os.path.join(args.data_dir, "stats.json")))
    tok = Tokenizer.from_file(os.path.join(args.data_dir, "tokenizer.json"))
    sp = stats["special_tokens"]
    eos = sp.get("<|endoftext|>", sp.get("<|eos|>", sp.get("</s>", sp.get("<eos>", 0))))
    bos = sp.get("<|bos|>", sp.get("<s>", sp.get("<bos>", eos)))
    sep = np.asarray(tok.encode(SEP_TEXT, add_special_tokens=False).ids, np.int64)

    if args.decode_only:                                            # take the method / reading from the trained run
        assert args.init_run, "--decode_only needs --init_run"
        ta = run_file(args, args.init_run)["args"]
        for k in ["base_run", "base_file", "method", "kd_teacher", "select", "prompt_len", "enc_len", "dec_len", "rank", "alpha",
                  "targets", "adapter_dim", "from_scratch"]:
            setattr(args, k, ta.get(k, 0))
    lm, cfg = load_lm(args, args.base_run, args.base_file, vars(args))
    arch, Lmax = cfg["arch"], cfg["max_seq_len"]
    if args.init_run:
        load_run_state(args, args.init_run, lm)
        log(f"loaded the trained weights of {args.init_run}")
    lm.to(dev)
    if arch == "encdec":
        P = min(args.enc_len, Lmax); S = min(args.dec_len, Lmax - 1); gen_budget = S
    else:
        P = min(args.prompt_len, Lmax // 2); S = Lmax - P - len(sep) - 1; gen_budget = Lmax - P - len(sep)
    gen_max = min(args.gen_max or gen_budget, gen_budget)
    gen_min = int(args.gen_min_frac * gen_max)
    total, trainable = count_params(lm)
    log(f"backbone {args.base_run} ({arch}, context {Lmax}) | method {args.method}{' + KD from ' + args.kd_teacher if args.kd_teacher else ''}"
        f" | select {args.select} | document tokens {P} | summary tokens {S} | generate {gen_min}-{gen_max} tokens, no-repeat {args.no_repeat}"
        f" | eos {eos} bos {bos} sep {sep.tolist()} | params {total / 1e6:.2f}M trainable {trainable / 1e6:.3f}M ({100 * trainable / total:.2f}%)")

    tr = Split(os.path.join(args.data_dir, "train.npz"), args.train_limit)
    va = Split(os.path.join(args.data_dir, "val.npz"), args.val_limit)
    te = Split(os.path.join(args.data_dir, "test.npz"), args.test_limit)

    def item(split, i):
        pr = split.select(i, args.select, P).astype(np.int64); s = split.summary(i).astype(np.int64)
        fits = len(s) <= S; s = s[:S]
        if arch == "encdec":
            inp = np.concatenate([[bos], s]).astype(np.int64); tgt = np.concatenate([s, [eos]]).astype(np.int64)
            m = np.ones(len(tgt), bool)
            if not fits:
                m[-1] = False                                       # cut summary: do not teach "stop here"
            return {"inp": inp, "tgt": tgt, "mask": m, "src": pr}
        seq = np.concatenate([pr, sep, s, [eos] if fits else []]).astype(np.int64)
        inp, tgt = seq[:-1], seq[1:]
        m = np.zeros(len(tgt), bool); m[len(pr) + len(sep) - 1:] = True    # loss on the summary tokens (+ end) only
        return {"inp": inp, "tgt": tgt, "mask": m}

    def batches(split, idxs, bs, rng=None):
        idxs = list(idxs)
        if rng is not None:
            rng.shuffle(idxs)
        if arch != "encdec":
            return [idxs[k:k + bs] for k in range(0, len(idxs), bs)]
        by = {}
        for i in idxs:
            by.setdefault(len(split.select(i, args.select, P)), []).append(i)
        out = [g[k:k + bs] for g in by.values() for k in range(0, len(g), bs)]
        if rng is not None:
            rng.shuffle(out)
        return out

    micro = {"bs": args.bs}

    @torch.no_grad()
    def val_loss():
        lm.eval(); tot = cnt = 0.0
        for b in batches(va, range(va.n), micro["bs"]):
            inp, tgt, m, src = make_batch([item(va, i) for i in b], arch, eos, dev)
            with amp():
                lg = sel_logits(lm, inp, m, src)
            tot += F.cross_entropy(lg, tgt[m], reduction="sum").item(); cnt += m.sum().item()
        lm.train()
        return tot / max(cnt, 1)

    # ---------------- training ----------------
    best = {"loss": float("inf"), "epoch": 0, "step": 0}
    train_min, steps, ep = 0.0, 0, 0
    trainable_names = [n for n, p_ in lm.named_parameters() if p_.requires_grad]
    if cuda:
        torch.cuda.reset_peak_memory_stats()
    if not args.decode_only:
        teacher = None
        if args.kd_teacher:
            ta = run_file(args, args.kd_teacher)["args"]
            tns = argparse.Namespace(**{**vars(args), **{k: ta[k] for k in ["rank", "alpha", "targets", "adapter_dim"]}, "method": ta["method"]})
            teacher, tcfg = load_lm(tns, ta["base_run"], ta["base_file"], {**vars(tns), "_teacher": 1})
            load_run_state(args, args.kd_teacher, teacher)
            assert tcfg["arch"] != "encdec" and arch != "encdec", "KD here pairs a decoder-only teacher and student"
            assert ta["prompt_len"] == args.prompt_len and ta["select"] == args.select and tcfg["vocab_size"] == cfg["vocab_size"], \
                "the teacher must read exactly the same input"
            teacher.to(dev).eval()
            for p_ in teacher.parameters():
                p_.requires_grad = False
            log(f"KD teacher {args.kd_teacher} ({tcfg['arch']} {count_params(teacher)[0] / 1e6:.1f}M params -> student {arch} "
                f"{total / 1e6:.1f}M) | loss = {args.kd_alpha} x gold-summary CE + {1 - args.kd_alpha:.2f} x T^2 KL(teacher || student), T={args.kd_T}")
        params = [p_ for p_ in lm.parameters() if p_.requires_grad]
        opt = torch.optim.AdamW(params, lr=args.lr if args.method == "full" else args.peft_lr, weight_decay=args.wd)
        base_lr = opt.param_groups[0]["lr"]
        scaler = torch.amp.GradScaler("cuda", enabled=cuda)
        rng = random.Random(args.seed)
        steps_per_epoch = len(batches(tr, range(tr.n), args.bs))
        total_steps = max(1, int(steps_per_epoch * args.epochs))
        if args.max_steps > 0:
            total_steps = min(total_steps, args.max_steps)
        warm = max(1, int(args.warmup * total_steps))
        budget = args.max_train_min * 60
        log(f"train {tr.n} docs, val {va.n}, test {te.n} | {steps_per_epoch} steps/epoch (batch {args.bs}) | planned {total_steps} steps | "
            f"lr {base_lr} | budget {args.max_train_min:.0f} min")
        best_state, t_train, stop = None, time.time(), False
        lm.train()
        while not stop:
            ep += 1
            for b in batches(tr, range(tr.n), args.bs, rng):
                lr_mult = min(1.0, (steps + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, steps / total_steps)))
                for g in opt.param_groups:
                    g["lr"] = base_lr * max(lr_mult, 0.02)
                items = [item(tr, i) for i in b]
                ntok = float(sum(int(it["mask"].sum()) for it in items))
                opt.zero_grad(set_to_none=True)
                try:
                    loss_v = 0.0
                    for k0 in range(0, len(items), micro["bs"]):          # micro-batches (only < bs after an out-of-memory)
                        inp, tgt, m, src = make_batch(items[k0:k0 + micro["bs"]], arch, eos, dev)
                        with amp():
                            lg = sel_logits(lm, inp, m, src)
                        loss = F.cross_entropy(lg, tgt[m], reduction="sum") / ntok          # mean over the whole batch's tokens
                        if teacher is not None:
                            with torch.no_grad(), amp():
                                tl = sel_logits(teacher, inp, m, src)
                            T = args.kd_T
                            kd = F.kl_div(F.log_softmax(lg / T, -1), F.softmax(tl / T, -1), reduction="sum") * T * T / ntok
                            loss = args.kd_alpha * loss + (1 - args.kd_alpha) * kd
                        scaler.scale(loss).backward()
                        loss_v += loss.item()
                except torch.cuda.OutOfMemoryError:
                    opt.zero_grad(set_to_none=True); lg = tl = kd = loss = inp = tgt = m = src = None
                    torch.cuda.empty_cache()
                    if micro["bs"] == 1:
                        raise
                    micro["bs"] = max(1, micro["bs"] // 2)
                    log(f"out of GPU memory -> micro-batches of {micro['bs']} (same batch of {args.bs} per update); batch skipped")
                    continue
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update()
                steps += 1
                if steps == 30 and args.max_steps <= 0:
                    sec = (time.time() - t_train) / steps
                    fit = int(max(50, 0.9 * budget / sec))
                    log(f"{sec:.2f} s/step -> {steps_per_epoch * sec / 60:.1f} min/epoch")
                    if fit < total_steps:
                        log(f"time budget allows {fit} of {total_steps} planned steps ({fit / steps_per_epoch:.1f} epochs)")
                        total_steps = fit
                if steps % 100 == 0:
                    log(f"epoch {ep} step {steps}/{total_steps} loss {loss_v:.4f} | {(time.time() - t_train) / 60:.1f} min")
                if steps >= total_steps or (time.time() - t_train) > budget:
                    stop = True; break
            vl = val_loss()
            log(f"epoch {ep} done (step {steps}) | validation loss {vl:.4f} (perplexity {math.exp(min(vl, 20)):.2f}) | {(time.time() - t_train) / 60:.1f} min")
            if vl < best["loss"] or best_state is None:
                best.update(loss=vl, epoch=ep, step=steps)
                sd = lm.state_dict(); best_state = {k: sd[k].detach().to("cpu", copy=True) for k in trainable_names}
        train_min = (time.time() - t_train) / 60
        sd = lm.state_dict(); sd.update(best_state); lm.load_state_dict(sd)
        teacher = opt = scaler = lg = loss = inp = tgt = m = src = None              # free training memory before generation
        tl = kd = None
        if cuda:
            torch.cuda.empty_cache()
        log(f"training done: {steps} steps, {ep} epochs, {train_min:.1f} min | best validation loss {best['loss']:.4f} (epoch {best['epoch']})")
        sd = lm.state_dict()
        torch.save({"state_dict": {k: sd[k].half() if sd[k].is_floating_point() else sd[k] for k in trainable_names},
                    "args": vars(args), "lm_config": cfg}, os.path.join(run_dir, "best.pt"))
        hf_up(args.hf_repo, os.path.join(run_dir, "best.pt"), f"phase6/runs/{args.run_name}/best.pt")   # early: KD teacher / re-decoding
    else:
        best["loss"] = val_loss()

    # ---------------- generation ----------------
    lm.eval()

    def summarise(split, idxs):
        groups = {}
        for i in idxs:
            pr = split.select(i, args.select, P).astype(np.int64)
            pr = pr if arch == "encdec" else np.concatenate([pr, sep])
            groups.setdefault(len(pr), []).append((i, pr))
        res = {}
        for L_, its in groups.items():
            for k in range(0, len(its), args.gen_bs):
                part = its[k:k + args.gen_bs]
                x = torch.as_tensor(np.stack([p_ for _, p_ in part])).to(dev)
                with amp():
                    g = generate(lm, x, gen_max, gen_min, eos, bos, args.no_repeat, args.rep_pen, args.temperature, args.top_p)
                for (i, _), row in zip(part, g):
                    row = row.tolist()
                    if eos in row:
                        row = row[:row.index(eos)]
                    res[i] = row
        return res

    def fallback(split, i):
        return tok.decode(split.document(i)[:gen_max].tolist()).strip() or "The court decided the case."

    t0 = time.time()
    gv = summarise(va, range(va.n))
    val_gen_s = time.time() - t0
    truth = json.load(open(os.path.join(args.data_dir, "val_truth.json")))
    vpred, reps, novs, lens, frec, fsup, empty_v = [], [], [], [], [], [], 0
    for i in range(va.n):
        txt = tok.decode(gv[i]).strip()
        if not txt:
            txt = fallback(va, i); empty_v += 1
        vpred.append({"id": va.name[i], "summary": txt})
        r_, n_ = ngram_stats(gv[i], va.document(i).tolist()); reps.append(r_); novs.append(n_); lens.append(len(txt.split()))
        a_, b_ = fact_stats(txt, truth[va.name[i]], tok.decode(va.document(i).tolist()))
        if a_ is not None: frec.append(a_)
        if b_ is not None: fsup.append(b_)
    vp = os.path.join(run_dir, "val_prediction.json"); json.dump(vpred, open(vp, "w"))
    tsub = os.path.join(run_dir, "val_truth_used.json")
    json.dump({x["id"]: truth[x["id"]] for x in vpred}, open(tsub, "w"))
    ev = official_eval(work, tsub, vp)
    log(f"VALIDATION (official evaluator, {va.n} held-out docs): BERTScore F1 {ev['bertscore_f1']} | ROUGE-1 {ev['rouge1']} | "
        f"ROUGE-L {ev['rougeL']} | {np.mean(lens):.0f} words/summary | repeated 3-grams {np.mean(reps):.3f} | novel 3-grams {np.mean(novs):.3f}"
        f" | numbers kept {np.mean(frec) if frec else 0:.3f} supported {np.mean(fsup) if fsup else 0:.3f} | empty {empty_v} | {val_gen_s:.0f}s")
    if ev["bertscore_f1"] is None:
        log("official evaluator output: " + ev["evaluator_output"])
    t1 = time.time()
    gt = summarise(te, range(te.n))
    test_gen_s = time.time() - t1
    preds, empty_t = [], 0
    for i in range(te.n):
        txt = tok.decode(gt[i]).strip()
        if not txt:                                                       # the evaluator rejects empty summaries
            txt = fallback(te, i); empty_t += 1
        preds.append({"id": te.name[i], "summary": txt})
    assert len(preds) == te.n and len(set(x["id"] for x in preds)) == te.n and all(x["summary"].strip() for x in preds)
    json.dump(preds, open(os.path.join(run_dir, "prediction.json"), "w"))
    with zipfile.ZipFile(os.path.join(run_dir, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(os.path.join(run_dir, "prediction.json"), arcname="prediction.json")
    adapter_mb = sum(p_.numel() for n, p_ in lm.named_parameters() if n in set(trainable_names)) * 2 / 1e6
    summary = {"run": args.run_name, "base_run": args.base_run, "arch": arch, "method": args.method + ("(scratch)" if args.from_scratch else ""), "kd_teacher": args.kd_teacher,
               "select": args.select, "doc_tokens": P, "summary_tokens_train": S, "gen_max": gen_max, "gen_min": gen_min,
               "decoding": {"no_repeat": args.no_repeat, "rep_pen": args.rep_pen, "temperature": args.temperature, "top_p": args.top_p},
               "decode_only": args.decode_only, "init_run": args.init_run, "params_total": total, "params_trainable": trainable,
               "saved_weights_mb": round(adapter_mb, 1),
               "val_bertscore_f1": ev["bertscore_f1"], "val_rouge1": ev["rouge1"], "val_rougeL": ev["rougeL"],
               "evaluator_output": ev["evaluator_output"], "val_loss": best["loss"], "best_epoch": best["epoch"], "best_step": best["step"],
               "steps": steps, "epochs_run": ep, "train_minutes": round(train_min, 1),
               "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if cuda else 0.0,
               "val_generation_seconds": round(val_gen_s, 1), "test_generation_seconds": round(test_gen_s, 1),
               "seconds_per_summary": round((val_gen_s + test_gen_s) / max(1, va.n + te.n), 3),
               "val_words_per_summary": float(np.mean(lens)), "val_repeated_3grams": float(np.mean(reps)),
               "val_novel_3grams": float(np.mean(novs)), "val_number_recall": float(np.mean(frec)) if frec else None,
               "val_number_supported": float(np.mean(fsup)) if fsup else None, "val_empty": empty_v, "test_empty": empty_t,
               "n_val": va.n, "n_test": te.n, "args": vars(args)}
    json.dump(summary, open(os.path.join(run_dir, "summary.json"), "w"), indent=1)
    log(f"TEST: {te.n} summaries (empty -> fallback: {empty_t}) -> {run_dir}/submission.zip")
    if args.hf_repo:
        for fn in ["prediction.json", "submission.zip", "val_prediction.json", "log.txt"]:
            hf_up(args.hf_repo, os.path.join(run_dir, fn), f"phase6/runs/{args.run_name}/{fn}")
        hf_up(args.hf_repo, os.path.join(run_dir, "summary.json"), f"phase6/runs/{args.run_name}/summary.json")
    log(f"DONE in {(time.time() - t_start) / 60:.1f} min")


if __name__ == "__main__":
    main()


Writing /kaggle/working/p6sum.py


In [5]:
%%writefile /kaggle/working/p6ens.py
"""CS-781 Phase 6: combine the summaries of several of our runs WITHOUT any outside model (consensus / MBR selection).
For every document, each run proposes one summary; we keep the proposal that agrees MOST with the other proposals
(mean unigram-overlap F1 = ROUGE-1-style agreement, computed by us). The idea (minimum-Bayes-risk selection): the summary
closest to the consensus is the one least likely to contain a run-specific mistake.
  ens_mbr3 / ens_mbr5 / ens_mbrall : the 3 / 5 / all runs with the best validation BERTScore
The validation score of every combination comes from the OFFICIAL evaluator on the same held-out split.
"""
import argparse, collections, json, os, re, shutil, zipfile

import p6sum as M

_W = re.compile(r"[a-z0-9]+")


def bag(t):
    return collections.Counter(_W.findall(t.lower()))


def f1(a, b):
    o = sum((a & b).values())
    if not o:
        return 0.0
    p, r = o / max(1, sum(a.values())), o / max(1, sum(b.values()))
    return 2 * p * r / (p + r)


def mbr(cands):
    """cands: list of summaries for one document -> index of the consensus summary."""
    bags = [bag(c) for c in cands]
    best, bi = -1.0, 0
    for i, b in enumerate(bags):
        s = sum(f1(b, o) for j, o in enumerate(bags) if j != i) / max(1, len(bags) - 1)
        if s > best:
            best, bi = s, i
    return bi


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--data_dir", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--hf_repo", default="")
    p.add_argument("--ks", default="3,5,0", help="0 = all runs")
    p.add_argument("--min_val", type=float, default=0.0)
    a = p.parse_args(argv)
    work = os.path.dirname(os.path.abspath(__file__))
    truth = json.load(open(os.path.join(a.data_dir, "val_truth.json")))

    names = set(x for x in os.listdir(a.out_dir) if os.path.isdir(os.path.join(a.out_dir, x)))
    if a.hf_repo:
        from huggingface_hub import HfApi
        for f in HfApi(token=os.environ.get("HF_TOKEN")).list_repo_files(a.hf_repo):
            if f.startswith("phase6/runs/") and f.endswith("/summary.json"):
                names.add(f.split("/")[2])
    runs = {}
    for n in sorted(names):
        if n.startswith("ens_") or n.startswith("_"):
            continue
        d = os.path.join(a.out_dir, n); os.makedirs(d, exist_ok=True)
        for fn in ["summary.json", "val_prediction.json", "prediction.json"]:
            if not os.path.exists(os.path.join(d, fn)):
                q = M.hf_get(a.hf_repo, f"phase6/runs/{n}/{fn}", os.path.join(a.out_dir, "_hub"))
                if q:
                    shutil.copy(q, os.path.join(d, fn))
        if not all(os.path.exists(os.path.join(d, fn)) for fn in ["summary.json", "val_prediction.json", "prediction.json"]):
            continue
        s = json.load(open(os.path.join(d, "summary.json")))
        if "method" not in s or "scratch" in s["method"] or s.get("val_bertscore_f1") is None or s["val_bertscore_f1"] < a.min_val:
            continue
        vp = {x["id"]: x["summary"] for x in json.load(open(os.path.join(d, "val_prediction.json")))}
        tp = {x["id"]: x["summary"] for x in json.load(open(os.path.join(d, "prediction.json")))}
        if set(vp) != set(truth):
            print("[ensemble] skip (validation ids differ):", n); continue
        runs[n] = (s, vp, tp)
    cands = sorted(runs, key=lambda n: -runs[n][0]["val_bertscore_f1"])
    print(f"[ensemble] {len(cands)} candidate runs:", ", ".join(f"{n}={runs[n][0]['val_bertscore_f1']:.4f}" for n in cands), flush=True)
    if len(cands) < 3:
        print("[ensemble] fewer than 3 runs - nothing to combine"); return
    test_ids = [x["id"] for x in json.load(open(os.path.join(a.out_dir, cands[0], "prediction.json")))]
    if not all(set(runs[n][2]) == set(test_ids) for n in cands):
        cands = [n for n in cands if set(runs[n][2]) == set(test_ids)]
    for K in sorted(set(int(k) for k in a.ks.split(",") if k.strip()), key=lambda k: k or 999):
        members = cands[:K] if K else cands
        if len(members) < 3 or (K and K > len(cands)):
            continue
        name = f"ens_mbr{K if K else 'all'}"
        rd = os.path.join(a.out_dir, name); os.makedirs(rd, exist_ok=True)
        picks = collections.Counter()
        vpred, tpred = [], []
        for split, ids, out, idx in [("val", list(truth), vpred, 1), ("test", test_ids, tpred, 2)]:
            for i in ids:
                c = [runs[m][idx][i] for m in members]
                j = mbr(c)
                out.append({"id": i, "summary": c[j]})
                if split == "val":
                    picks[members[j]] += 1
        assert all(x["summary"].strip() for x in tpred) and len(set(x["id"] for x in tpred)) == len(test_ids)
        json.dump(vpred, open(os.path.join(rd, "val_prediction.json"), "w"))
        json.dump(tpred, open(os.path.join(rd, "prediction.json"), "w"))
        with zipfile.ZipFile(os.path.join(rd, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as zf:
            zf.write(os.path.join(rd, "prediction.json"), arcname="prediction.json")
        json.dump(truth, open(os.path.join(rd, "val_truth_used.json"), "w"))
        ev = M.official_eval(work, os.path.join(rd, "val_truth_used.json"), os.path.join(rd, "val_prediction.json"))
        info = {"members": members, "picked_on_val": dict(picks), "val_bertscore_f1": ev["bertscore_f1"], "val_rouge1": ev["rouge1"],
                "val_rougeL": ev["rougeL"], "evaluator_output": ev["evaluator_output"],
                "best_member": cands[0], "best_member_val_bertscore_f1": runs[cands[0]][0]["val_bertscore_f1"]}
        json.dump(info, open(os.path.join(rd, "summary.json"), "w"), indent=1)
        print(f"[ensemble] {name}: {len(members)} runs | VALIDATION BERTScore F1 {ev['bertscore_f1']} ROUGE-1 {ev['rouge1']} ROUGE-L {ev['rougeL']}"
              f" (best single {runs[cands[0]][0]['val_bertscore_f1']:.4f}) | chosen per document: {dict(picks.most_common())}", flush=True)
        if a.hf_repo:
            for fn in ["prediction.json", "submission.zip", "val_prediction.json", "summary.json"]:
                M.hf_up(a.hf_repo, os.path.join(rd, fn), f"phase6/runs/{name}/{fn}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/p6ens.py


In [6]:
%%writefile /kaggle/working/evaluate_phase6.py
#!/usr/bin/env python3
"""Local evaluator for Phase 6.

Requires: rouge-score, bert-score, torch.
Truth JSON: {"doc_id":"reference summary", ...}
Prediction JSON: either {"doc_id":"summary", ...} or [{"id":"...","summary":"..."}, ...]
"""
import argparse,json,sys
try:
    from rouge_score import tokenizers, rouge_scorer
    import bert_score
except ImportError as e:
    print('Missing dependency. Install requirements.txt first.', file=sys.stderr); raise

def load(path):
    with open(path,encoding='utf-8') as f: return json.load(f)

def lcs(a,b):
    if len(a)>len(b): a,b=b,a
    masks={}
    for i,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<i)
    s=0
    for t in b:
        m=masks.get(t,0); x=m|s; y=(s<<1)|1; s=x & ~(x-y)
    return s.bit_count() if hasattr(int,'bit_count') else bin(s).count('1')

def read_pred(x):
    if isinstance(x,dict): out=x
    elif isinstance(x,list):
        out={}
        for r in x:
            if not isinstance(r,dict) or 'id' not in r or 'summary' not in r: raise ValueError('Each prediction item must contain id and summary.')
            if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
            out[r['id']]=r['summary']
    else: raise ValueError('Prediction must be a JSON object or list of objects.')
    for k,v in out.items():
        if not isinstance(k,str) or not isinstance(v,str) or not v.strip(): raise ValueError(f'Invalid summary for ID {k!r}.')
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 6 summaries on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); a=ap.parse_args()
    t=load(a.truth); p=read_pred(load(a.prediction))
    if not isinstance(t,dict) or not t: raise ValueError('Truth must be a non-empty JSON object.')
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    preds=[p[k].strip() for k in t]; refs=[t[k].strip() for k in t]
    scorer=rouge_scorer.RougeScorer(['rouge1'],use_stemmer=True); tok=tokenizers.DefaultTokenizer(use_stemmer=True)
    r1=rl=0.0
    for pr,rf in zip(preds,refs):
        s=scorer.score(rf,pr); r1+=s['rouge1'].fmeasure
        pt=tok.tokenize(pr); rt=tok.tokenize(rf)
        if not pt and not rt: x=1.0
        elif not pt or not rt: x=0.0
        else:
            z=lcs(pt,rt); pp=z/len(pt); rr=z/len(rt); x=0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)
        rl+=x
    n=len(t); r1/=n; rl/=n
    try:
        import torch; device='cuda' if torch.cuda.is_available() else 'cpu'
    except ImportError: device=None
    _,_,f=bert_score.score(preds,refs,lang='en',batch_size=8,verbose=False,**({'device':device} if device else {}))
    print(f'Documents: {n}'); print(f'BERTScore F1: {float(f.mean().item()):.6f}'); print(f'ROUGE-1: {r1:.6f}'); print(f'ROUGE-L: {rl:.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)


Writing /kaggle/working/evaluate_phase6.py


In [7]:
# ===================== P6 CELL 3 - Data: normalise + split into sentences + tokenise ONCE with our tokenizer (cached on HF) =====================
# First session: downloads the official Phase_6_Summ files, holds out 200 training documents as a VALIDATION split
# (the task gives no dev set), tokenises everything and uploads the cache. Later sessions just download the cache.
sys.path.insert(0, WORK)
hub_files = set(api.list_repo_files(CKPT_REPO))
cache = ["train.npz", "val.npz", "test.npz", "stats.json", "val_truth.json"]
if all(f"phase6/data/{f}" in hub_files for f in cache):
    for f in cache:
        shutil.copy(hf_hub_download(CKPT_REPO, f"phase6/data/{f}", repo_type="model", token=HF_TOKEN, local_dir=f"{P6}/hub"), f"{DATA}/{f}")
    print("token cache downloaded from Hugging Face")
else:
    t0 = time.time()
    for p in [x.path for x in api.list_repo_tree(SRC_REPO, repo_type="dataset", path_in_repo="Phase_6_Summ", recursive=True)
              if getattr(x, "size", None) is not None]:
        lp = hf_hub_download(SRC_REPO, p, repo_type="dataset", token=HF_TOKEN, local_dir=f"{P6}/raw")
        shutil.copy(lp, f"{RAW}/{p.split('/')[-1]}")
    missing = [n for n in ["train.json", "test.json"] if not os.path.exists(f"{RAW}/{n}")]
    assert not missing, f"missing files {missing} (found {sorted(os.listdir(RAW))}) - paste this output to Claude"
    print("Phase-6 files:", {f: f"{os.path.getsize(f'{RAW}/{f}') / 1e6:.1f} MB" for f in sorted(os.listdir(RAW))}, f"| {time.time() - t0:.0f}s")
    r = subprocess.run([sys.executable, f"{WORK}/p6data.py", "--raw_dir", RAW, "--tok", TOK_PATH, "--out_dir", DATA], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    assert r.returncode == 0, "data preparation failed - paste this output to Claude"
    for f in cache:
        api.upload_file(path_or_fileobj=f"{DATA}/{f}", path_in_repo=f"phase6/data/{f}", repo_id=CKPT_REPO, repo_type="model")
    print(f"tokenised + uploaded in {(time.time() - t0) / 60:.1f} min")
    shutil.rmtree(f"{P6}/raw", ignore_errors=True)
stats = json.load(open(f"{DATA}/stats.json"))
print("records:", {k: stats[k] for k in ["train_records", "train_kept", "dropped_duplicate_or_empty", "n_train", "n_val", "n_test", "test_records"]})
for sp in ["train", "val", "test"]:
    s = stats[sp]
    line = (f"{sp:5s}: {s['docs']:5d} docs | document tokens mean {s['doc_tokens_mean']:.0f} median {s['doc_tokens_median']:.0f} p90 {s['doc_tokens_p90']:.0f}"
            f" | sentences/doc {s['sentences_per_doc']:.0f}")
    if "summary_tokens_mean" in s:
        line += (f" | summary tokens mean {s['summary_tokens_mean']:.0f} median {s['summary_tokens_median']:.0f} (p10 {s['summary_tokens_p10']:.0f}, "
                 f"p90 {s['summary_tokens_p90']:.0f}) = {s['summary_words_mean']:.0f} words | summaries > 512 tokens: {100 * s['frac_summary_over_512_tokens']:.0f}%")
    print(line)
print("special tokens:", stats["special_tokens"])


test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

Phase-6 files: {'test.json': '2.7 MB', 'train.json': '210.6 MB'} | 3s
train {"docs": 6828, "doc_tokens_mean": 5087.510251903925, "doc_tokens_median": 3656.5, "doc_tokens_p90": 9091.000000000002, "sentences_per_doc": 177.09226713532513, "summary_tokens_mean": 994.1211189220855, "summary_tokens_median": 750.0, "summary_tokens_p10": 341.0, "summary_tokens_p90": 1774.3000000000002, "summary_words_mean": 842.2086994727592, "summary_words_median": 639.0, "frac_summary_over_512_tokens": 0.7322788517867604}
val {"docs": 200, "doc_tokens_mean": 4775.66, "doc_tokens_median": 3591.0, "doc_tokens_p90": 8904.3, "sentences_per_doc": 165.0, "summary_tokens_mean": 904.885, "summary_tokens_median": 713.5, "summary_tokens_p10": 329.7, "summary_tokens_p90": 1646.0, "summary_words_mean": 764.485, "summary_words_median": 613.5, "frac_summary_over_512_tokens": 0.705}
test {"docs": 100, "doc_tokens_mean": 5556.87, "doc_tokens_median": 3763.0, "doc_tokens_p90": 10258.10000000001, "sentences_per_doc": 191.8}
s

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

tokenised + uploaded in 2.2 min
records: {'train_records': 7030, 'train_kept': 7028, 'dropped_duplicate_or_empty': 2, 'n_train': 6828, 'n_val': 200, 'n_test': 100, 'test_records': 100}
train:  6828 docs | document tokens mean 5088 median 3656 p90 9091 | sentences/doc 177 | summary tokens mean 994 median 750 (p10 341, p90 1774) = 842 words | summaries > 512 tokens: 73%
val  :   200 docs | document tokens mean 4776 median 3591 p90 8904 | sentences/doc 165 | summary tokens mean 905 median 714 (p10 330, p90 1646) = 764 words | summaries > 512 tokens: 70%
test :   100 docs | document tokens mean 5557 median 3763 p90 10258 | sentences/doc 192
special tokens: {'<|endoftext|>': 0, '<|bos|>': 2, '<|pad|>': 1, '<|sep|>': 3}


In [8]:
# ===================== P6 CELL 7 - Run the experiments on both GPUs (queue with KD / re-decoding dependencies, resumable) =====================
# Every run trains until its TIME BUDGET (max_train_min) or the planned epochs, checks the validation loss after every epoch
# and keeps the best weights; then it writes summaries for the 200 held-out validation documents, scores them with the
# OFFICIAL evaluate_phase6.py (BERTScore F1 / ROUGE-1 / ROUGE-L) and writes the test summaries + submission.zip.
# Finished runs (phase6/runs/<run>/summary.json on HF) are skipped, so a new session continues where the last one stopped.
import threading
CK = "/tmp/cs781/ckpt"
OUT = f"{WORK}/p6runs"; os.makedirs(OUT, exist_ok=True)
for run, fn in [("dapt_llama", "best_model.pt"), ("explore_gpt2", "final_model.pt"),
                ("explore_encdec", "final_model.pt"), ("explore_shared", "final_model.pt")]:
    hf_hub_download(CKPT_REPO, f"runs/{run}/{fn}", repo_type="model", local_dir=CK, token=HF_TOKEN)
print("Phase-1 checkpoints ready")

COMMON = dict(bs=8, epochs=6, max_train_min=30, prompt_len=512, enc_len=896, dec_len=640, lr=5e-5, peft_lr=5e-4,
              rank=16, alpha=32, adapter_dim=64, no_repeat=4, gen_bs=32)
LLAMA  = dict(base_run="dapt_llama", base_file="best_model.pt")
GPT2   = dict(base_run="explore_gpt2", base_file="final_model.pt")
ENCDEC = dict(base_run="explore_encdec", base_file="final_model.pt")
SHARED = dict(base_run="explore_shared", base_file="final_model.pt")
TEACHER = "sm_llama_lorma_lead"
RUNS = [   # priority order (2 GPUs take jobs from the top)
    (TEACHER,                   {**LLAMA,  "method": "lorma"}),                                   # LoRMA (compulsory) + KD teacher
    ("sm_encdec_lorma_lead",    {**ENCDEC, "method": "lorma"}),                                   # encoder reads 896 document tokens
    ("sm_gpt2_lorma_lead",      {**GPT2,   "method": "lorma"}),
    ("sm_shared_kd_lead",       {**SHARED, "method": "full", "lr": 1e-4, "kd_teacher": TEACHER}),  # KD: llama (108M) -> shared (59M)
    ("sm_llama_lora_lead",      {**LLAMA,  "method": "lora"}),                                    # LoRA
    ("sm_llama_houlsby_lead",   {**LLAMA,  "method": "houlsby", "peft_lr": 1e-3}),                # Houlsby adapters
    ("sm_llama_pfeiffer_lead",  {**LLAMA,  "method": "pfeiffer", "peft_lr": 1e-3}),               # Pfeiffer adapters
    ("sm_llama_lorma_central",  {**LLAMA,  "method": "lorma", "select": "central"}),              # long documents: central sentences
    ("sm_llama_lorma_leadcentral", {**LLAMA, "method": "lorma", "select": "leadcentral"}),        # long documents: lead + central
    ("sm_llama_full_lead",      {**LLAMA,  "method": "full"}),                                    # full fine-tuning (reference)
    ("sm_shared_full_lead",     {**SHARED, "method": "full", "lr": 1e-4}),                        # same student WITHOUT KD
]
DEC = dict(LLAMA, decode_only=1, init_run=TEACHER)      # decoding study on the LoRMA run: no training, only generation
RUNS += [
    ("dec_lorma_len128",  {**DEC, "gen_max": 128}),     # generation length vs BERTScore / ROUGE
    ("dec_lorma_len256",  {**DEC, "gen_max": 256}),
    ("dec_lorma_len384",  {**DEC, "gen_max": 384}),
    ("dec_lorma_nr0",     {**DEC, "no_repeat": 0}),     # repetition control: no n-gram blocking
    ("dec_lorma_nr3",     {**DEC, "no_repeat": 3}),
    ("dec_lorma_nr6",     {**DEC, "no_repeat": 6}),
    ("dec_lorma_free",    {**DEC, "gen_min_frac": 0.2}), # the model decides the length (may stop after 20% of the budget)
    ("sm_llama_scratch_lead", {**LLAMA, "method": "full", "lr": 3e-4, "from_scratch": 1}),   # ablation: NO Phase-1 weights
]
ONLY = None          # e.g. ["sm_llama_lorma_lead"] to run a subset

def p6_cmd(name, cfg, out_dir, hf_repo, extra=()):
    cmd = [sys.executable, f"{WORK}/p6sum.py", "--run_name", name, "--data_dir", DATA, "--ckpt_dir", CK,
           "--out_dir", out_dir, "--hf_repo", hf_repo]
    for k, v in {**COMMON, **cfg}.items():
        cmd += [f"--{k}", str(v)]
    return cmd + list(extra)

# ---- safety test: 3 training steps + validation summaries scored by the OFFICIAL evaluator + FULL test prediction + zip
#      on the REAL data (short summaries to be quick); nothing uploaded ----
t_s = time.time()
smoke = p6_cmd("_smoke", {**SHARED, "method": "lorma", "max_steps": 3, "gen_max": 24},
               f"{P6}/smoke", "", ["--train_limit", "32", "--val_limit", "16"])
r = subprocess.run(smoke, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"), capture_output=True, text=True)
print("\n".join([l for l in (r.stdout + r.stderr).splitlines() if "][_smoke]" in l][-8:]))
sm = f"{P6}/smoke/_smoke/summary.json"
ok_smoke = r.returncode == 0 and os.path.exists(f"{P6}/smoke/_smoke/submission.zip") and os.path.exists(sm) \
           and json.load(open(sm))["val_bertscore_f1"] is not None
if not ok_smoke:
    print("\n".join((r.stdout + r.stderr).splitlines()[-30:]))
    if os.path.exists(sm): print("evaluator:", json.load(open(sm))["evaluator_output"])
    raise RuntimeError("SAFETY TEST FAILED - nothing was trained. Send this output to Claude.")
print(f"safety test passed in {time.time() - t_s:.0f}s (official evaluator ran: {json.load(open(sm))['evaluator_output']})\n")
if torch.cuda.device_count() == 0:
    print("CPU session: switch Accelerator to GPU T4 x2 and run again."); RUNS = []

hub_files = set(api.list_repo_files(CKPT_REPO))
done, failed, running, lock = set(), set(), set(), threading.Lock()
pending = [(n, c) for n, c in RUNS if ONLY is None or n in ONLY]
for n, c in list(pending):
    if f"phase6/runs/{n}/summary.json" in hub_files:          # finished in an earlier session -> skip
        os.makedirs(f"{OUT}/{n}", exist_ok=True)
        for fn in ["summary.json", "submission.zip", "prediction.json", "val_prediction.json", "log.txt"]:
            if f"phase6/runs/{n}/{fn}" in hub_files:
                p = hf_hub_download(CKPT_REPO, f"phase6/runs/{n}/{fn}", repo_type="model", local_dir=f"{P6}/hub", token=HF_TOKEN)
                shutil.copy(p, f"{OUT}/{n}/{fn}")
        done.add(n); pending.remove((n, c)); print("already done:", n)

def dep_of(c):
    return c.get("kd_teacher") or c.get("init_run")

def next_job():
    with lock:
        for i, (n, c) in enumerate(pending):
            t = dep_of(c)
            if t and t not in done and t not in running and all(t != m for m, _ in pending):
                if f"phase6/runs/{t}/best.pt" in hub_files: done.add(t)      # trained in an earlier session
                else: failed.add(t)
            if t and t in failed:
                pending.pop(i); failed.add(n); print(f"SKIP {n}: {t} failed / not available", flush=True); return "AGAIN"
            if not t or t in done:
                running.add(n); return pending.pop(i)
        return "WAIT" if pending else None

def worker(gpu):
    while True:
        job = next_job()
        if job is None: return
        if job in ("WAIT", "AGAIN"):
            time.sleep(30 if job == "WAIT" else 0); continue
        name, cfg = job
        with open(f"{OUT}/{name}.out", "w") as f:
            r = subprocess.run(p6_cmd(name, cfg, OUT, CKPT_REPO), env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)),
                               stdout=f, stderr=subprocess.STDOUT)
        ok_ = r.returncode == 0 and os.path.exists(f"{OUT}/{name}/summary.json")
        with lock:
            (done if ok_ else failed).add(name); running.discard(name)
            print(f"{'FINISHED' if ok_ else 'FAILED  '} {name} on GPU {gpu}", flush=True)
            if not ok_:
                print(f"===== last lines of {name}.out =====\n" + "\n".join(open(f"{OUT}/{name}.out").read().splitlines()[-30:]), flush=True)

def status_line(name):
    p = f"{OUT}/{name}.out"
    if not os.path.exists(p): return "(waiting)"
    keep = [l for l in open(p).read().splitlines() if "step" in l or "VALIDATION" in l or "Error" in l or "DONE" in l]
    return keep[-1][:200] if keep else "(starting)"

threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
for t in threads: t.start()
t0 = time.time()
while any(t.is_alive() for t in threads):
    for t in threads: t.join(timeout=300 / len(threads))
    print(f"--- {(time.time() - t0) / 60:.0f} min | done {sorted(done)} | failed {sorted(failed)}", flush=True)
    for n, _ in RUNS:
        if n not in done and n not in failed and (ONLY is None or n in ONLY): print(f"   {n:28s} {status_line(n)}", flush=True)
print("ALL FINISHED. done:", sorted(done), "| failed:", sorted(failed))


runs/dapt_llama/best_model.pt: reconstructing file:   0%|          |  0.00B /  431MB            

runs/dapt_llama/best_model.pt: downloading bytes:           |  0.00B            

runs/explore_gpt2/final_model.pt: reconstructing file:   0%|          |  0.00B /  444MB            

runs/explore_gpt2/final_model.pt: downloading bytes:           |  0.00B            

runs/explore_encdec/final_model.pt: reconstructing file:   0%|          |  0.00B /  469MB            

runs/explore_encdec/final_model.pt: downloading bytes:           |  0.00B            

runs/explore_shared/final_model.pt: reconstructing file:   0%|          |  0.00B /  235MB            

runs/explore_shared/final_model.pt: downloading bytes:           |  0.00B            

Phase-1 checkpoints ready
[05:01:46][_smoke] backbone explore_shared (shared, context 1024) | method lorma | select lead | document tokens 512 | summary tokens 509 | generate 19-24 tokens, no-repeat 4 | eos 0 bos 2 sep [19169, 41] | params 60.38M trainable 1.647M (2.73%)
[05:01:47][_smoke] train 32 docs, val 16, test 100 | 4 steps/epoch (batch 8) | planned 3 steps | lr 0.0005 | budget 30 min
[05:01:50][_smoke] epoch 1 done (step 3) | validation loss 4.0206 (perplexity 55.74) | 0.1 min
[05:01:50][_smoke] training done: 3 steps, 1 epochs, 0.1 min | best validation loss 4.0206 (epoch 1)
[05:02:18][_smoke] VALIDATION (official evaluator, 16 held-out docs): BERTScore F1 0.807534 | ROUGE-1 0.066975 | ROUGE-L 0.058937 | 20 words/summary | repeated 3-grams 0.060 | novel 3-grams 0.864 | numbers kept 0.042 supported 0.646 | empty 0 | 1s
[05:02:21][_smoke] TEST: 100 summaries (empty -> fallback: 0) -> /tmp/cs781/p6/smoke/_smoke/submission.zip
[05:02:21][_smoke] DONE in 0.6 min
safety test passed 

In [9]:
# ===================== P6 CELL 8 - Results (official evaluator on the held-out validation split), baselines, analysis, zips =====================
import numpy as np
ZIPS = f"{WORK}/zips"; os.makedirs(ZIPS, exist_ok=True)
rows = []
for n, _ in RUNS if RUNS else []:
    p = f"{OUT}/{n}/summary.json"
    if not os.path.exists(p): continue
    s = json.load(open(p))
    if s.get("val_bertscore_f1") is None: print("no validation score:", n, s.get("evaluator_output")); continue
    rows.append(s)
    if os.path.exists(f"{OUT}/{n}/submission.zip"):
        shutil.copy(f"{OUT}/{n}/submission.zip", f"{ZIPS}/{n}.zip")
rows.sort(key=lambda s: -s["val_bertscore_f1"])
print(f"VALIDATION = {rows[0]['n_val'] if rows else '?'} held-out training documents (never trained on), scored by the OFFICIAL evaluate_phase6.py\n")
print(f"{'run':28s} {'model':6s} {'method':12s} {'reading':11s} {'trainable':>9s} {'BERTSc':>6s} {'ROUGE1':>6s} {'ROUGEL':>6s} {'words':>5s} "
      f"{'rep3':>5s} {'novel3':>6s} {'num-rec':>7s} {'num-sup':>7s} {'valloss':>7s} {'ep':>4s} {'train':>6s} {'GPU':>5s} {'s/summ':>6s}  decoding")
for s in rows:
    m = s["method"] + ("+KD" if s.get("kd_teacher") else "")
    dec = f"max {s['gen_max']} tok, no-repeat {s['decoding']['no_repeat']}" + (" (re-decoded)" if s.get("decode_only") else "")
    print(f"{s['run']:28s} {s['arch']:6s} {m:12s} {s['select']:11s} {s['params_trainable'] / 1e6:8.2f}M {s['val_bertscore_f1']:6.4f} {s['val_rouge1']:6.4f} "
          f"{s['val_rougeL']:6.4f} {s['val_words_per_summary']:5.0f} {s['val_repeated_3grams']:5.3f} {s['val_novel_3grams']:6.3f} "
          f"{s['val_number_recall'] or 0:7.3f} {s['val_number_supported'] or 0:7.3f} {s['val_loss']:7.4f} {s['best_epoch']:>2d}/{s['epochs_run']:<2d} "
          f"{s['train_minutes']:5.1f}m {s['peak_gpu_gb']:5.2f} {s['seconds_per_summary']:6.2f}  {dec}")
print("\nBERTSc / ROUGE1 / ROUGEL = official metrics | words = mean words per generated summary | rep3 = share of repeated 3-grams inside a"
      " summary (redundancy) | novel3 = share of summary 3-grams NOT copied from the document (abstractiveness) | num-rec = share of the"
      " reference's numbers (sections, years, amounts) present in the summary | num-sup = share of the summary's numbers that occur in"
      " the document (1 - num-sup ~ invented numbers) | ep = best epoch / epochs run | s/summ = generation seconds per summary")

# ---- reference points: extractive LEAD baseline (first N words of the document) and the length of the references ----
truth = json.load(open(f"{DATA}/val_truth.json"))
ref_words = [len(v.split()) for v in truth.values()]
print(f"\nreference summaries (validation): mean {np.mean(ref_words):.0f} words, median {np.median(ref_words):.0f}"
      f" | BERTScore reads only the first ~510 roberta tokens (~400 words) of each text")
raw_val = None
if os.path.exists(f"{RAW}/train.json"):
    sys.path.insert(0, WORK); import p6data
    raw_val = {i: p6data.normalize(d) for i, d, _ in p6data.load_records(f"{RAW}/train.json") if i in truth}
else:
    import p6sum as _M
    _va = _M.Split(f"{DATA}/val.npz")
    raw_val = {_va.name[i]: tok.decode(_va.document(i)[:900].tolist()) for i in range(_va.n)}
base_rows = []
for nw in [150, 300, 400]:
    bd = f"{P6}/baseline_lead{nw}"; os.makedirs(bd, exist_ok=True)
    json.dump([{"id": k, "summary": " ".join(raw_val[k].split()[:nw]) or "."} for k in truth], open(f"{bd}/val_prediction.json", "w"))
    json.dump(truth, open(f"{bd}/truth.json", "w"))
    r = subprocess.run([sys.executable, f"{WORK}/evaluate_phase6.py", "--truth", f"{bd}/truth.json", "--prediction", f"{bd}/val_prediction.json"],
                       capture_output=True, text=True)
    print(f"   extractive baseline: first {nw} words of the document  ->  " + " | ".join(l.strip() for l in (r.stdout + r.stderr).splitlines()[-3:]))
    base_rows.append({"baseline": f"lead{nw}", "evaluator_output": (r.stdout + r.stderr).strip()})

# ---- best run per Phase-1 model (what to submit) ----
if rows:
    print("\nBEST RUN PER PHASE-1 MODEL (validation BERTScore F1):")
    for a_ in ["llama", "gpt2", "encdec", "shared"]:
        c = [s for s in rows if s["arch"] == a_ and "scratch" not in s["method"]]
        if c:
            b = c[0]
            print(f"   {a_:6s}: {b['run']:28s} BERTScore {b['val_bertscore_f1']:.4f} ROUGE-1 {b['val_rouge1']:.4f} ROUGE-L {b['val_rougeL']:.4f}"
                  f" ({b['method']}{'+KD' if b.get('kd_teacher') else ''}) -> zips/{b['run']}.zip")
    b = rows[0]
    vp = {x["id"]: x["summary"] for x in json.load(open(f"{OUT}/{b['run']}/val_prediction.json"))}
    k = sorted(truth)[0]
    print(f"\nexample ({b['run']}, validation document {k}):\n   GENERATED: {vp[k][:700]}\n   REFERENCE: {truth[k][:700]}")
json.dump({"runs": rows, "baselines": base_rows}, open(f"{P6}/p6_results.json", "w"), indent=1)
api.upload_file(path_or_fileobj=f"{P6}/p6_results.json", path_in_repo="phase6/results/p6_results.json", repo_id=CKPT_REPO, repo_type="model")
print("\nresults table uploaded to Hugging Face: phase6/results/p6_results.json")
print("zips ready in", ZIPS, ":", sorted(os.listdir(ZIPS)))


VALIDATION = 200 held-out training documents (never trained on), scored by the OFFICIAL evaluate_phase6.py

run                          model  method       reading     trainable BERTSc ROUGE1 ROUGEL words  rep3 novel3 num-rec num-sup valloss   ep  train   GPU s/summ  decoding
dec_lorma_len256             llama  lorma        lead            2.85M 0.8388 0.3530 0.2078   223 0.121  0.638   0.191   0.788  2.0011  0/0    0.0m  1.80   0.28  max 256 tok, no-repeat 4 (re-decoded)
sm_llama_lorma_central       llama  lorma        central         2.85M 0.8371 0.4464 0.2339   400 0.161  0.699   0.262   0.642  1.9974  2/4   26.8m  7.63   0.54  max 510 tok, no-repeat 4
dec_lorma_len128             llama  lorma        lead            2.85M 0.8370 0.2413 0.1572   112 0.073  0.527   0.115   0.849  2.0011  0/0    0.0m  1.80   0.14  max 128 tok, no-repeat 4 (re-decoded)
dec_lorma_len384             llama  lorma        lead            2.85M 0.8363 0.3950 0.2226   322 0.150  0.701   0.239   0.742  2.0011 

In [10]:
# ===================== P6 CELL 9 - Round 2: best decoding for every model's best run + consensus (MBR) combinations =====================
# (a) The decoding study (dec_* runs) re-decoded the LoRMA llama run with other length / repetition settings. If one of them
#     beat the default on validation BERTScore, the best run of EVERY Phase-1 model is re-decoded with that setting (no training).
# (b) ens_mbr*: per document, keep the summary that agrees most with the other runs' summaries (no outside model).
import threading
rows = sorted([json.load(open(f"{OUT}/{n}/summary.json")) for n in os.listdir(OUT)
               if os.path.exists(f"{OUT}/{n}/summary.json") and not n.startswith(("ens_", "_"))
               and json.load(open(f"{OUT}/{n}/summary.json")).get("val_bertscore_f1") is not None], key=lambda s: -s["val_bertscore_f1"])
base = next((s for s in rows if s["run"] == TEACHER), None)
study = [s for s in rows if s["run"].startswith("dec_lorma_")]
jobs = []
if base and study:
    bs_ = max(study, key=lambda s: s["val_bertscore_f1"])
    print(f"decoding study (validation BERTScore F1): default {base['val_bertscore_f1']:.4f} (max {base['gen_max']} tokens, no-repeat "
          f"{base['decoding']['no_repeat']}) | " + " | ".join(f"{s['run'][10:]} {s['val_bertscore_f1']:.4f}" for s in sorted(study, key=lambda s: s['run'])))
    if bs_["val_bertscore_f1"] > base["val_bertscore_f1"] + 0.001:
        setting = {"gen_max": bs_["args"]["gen_max"], "no_repeat": bs_["decoding"]["no_repeat"], "gen_min_frac": bs_["args"]["gen_min_frac"]}
        print(f"-> best decoding: {bs_['run']} {setting}; re-decoding the best run of every Phase-1 model with it")
        for a_ in ["llama", "gpt2", "encdec", "shared"]:
            c = [s for s in rows if s["arch"] == a_ and not s.get("decode_only") and "scratch" not in s["method"]]
            if c:
                b = c[0]; g = setting["gen_max"]
                if b["arch"] == "encdec" and g:                  # encdec summary budget differs: keep the same share of it
                    g = int(round(g / max(base["gen_max"], 1) * b["gen_max"]))
                name = f"r2_{b['run'][3:]}_dec"
                if not os.path.exists(f"{OUT}/{name}/summary.json"):
                    jobs.append((name, {"base_run": b["base_run"], "base_file": b["args"]["base_file"], "decode_only": 1, "init_run": b["run"],
                                        "gen_max": g, "no_repeat": setting["no_repeat"], "gen_min_frac": setting["gen_min_frac"]}))
    else:
        print("-> the default decoding is already the best: nothing to re-decode")
if jobs:
    q = list(jobs); ql = threading.Lock()
    def w2(gpu):
        while True:
            with ql:
                if not q: return
                name, cfg = q.pop(0)
            with open(f"{OUT}/{name}.out", "w") as f:
                subprocess.run(p6_cmd(name, cfg, OUT, CKPT_REPO), env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)), stdout=f, stderr=subprocess.STDOUT)
            print(("FINISHED " if os.path.exists(f"{OUT}/{name}/summary.json") else "FAILED ") + name, "|", status_line(name), flush=True)
    th = [threading.Thread(target=w2, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
    for t in th: t.start()
    for t in th: t.join()
    RUNS = RUNS + [(n, c) for n, c in jobs if (n, c) not in RUNS]

r = subprocess.run([sys.executable, f"{WORK}/p6ens.py", "--data_dir", DATA, "--out_dir", OUT, "--hf_repo", CKPT_REPO], capture_output=True, text=True)
print("\n".join([l for l in (r.stdout + r.stderr).splitlines() if "[ensemble]" in l or "Error" in l or "Traceback" in l][-20:]))
if r.returncode != 0:
    print("\n".join((r.stdout + r.stderr).splitlines()[-30:]))
for o in sorted(os.listdir(OUT)):
    if (o.startswith("ens_") or o.startswith("r2_")) and os.path.exists(f"{OUT}/{o}/submission.zip"):
        shutil.copy(f"{OUT}/{o}/submission.zip", f"{ZIPS}/{o}.zip")

print("\nFINAL RANKING (validation BERTScore F1, official evaluator):")
allr = []
for n in os.listdir(OUT):
    p = f"{OUT}/{n}/summary.json"
    if os.path.exists(p) and not n.startswith("_"):
        s = json.load(open(p))
        if s.get("val_bertscore_f1") is not None:
            allr.append((s["val_bertscore_f1"], s.get("val_rouge1"), s.get("val_rougeL"), n, s.get("arch", "mixed")))
for v, r1, rl, n, a_ in sorted(allr, reverse=True)[:20]:
    print(f"   {n:30s} {a_:6s} BERTScore {v:.4f} | ROUGE-1 {r1:.4f} | ROUGE-L {rl:.4f}")
print("zips:", sorted(os.listdir(ZIPS)))


decoding study (validation BERTScore F1): default 0.8328 (max 510 tokens, no-repeat 4) | free 0.8348 | len128 0.8370 | len256 0.8388 | len384 0.8363 | nr0 0.7992 | nr3 0.8355 | nr6 0.8275
-> best decoding: dec_lorma_len256 {'gen_max': 256, 'no_repeat': 4, 'gen_min_frac': 0.8}; re-decoding the best run of every Phase-1 model with it
FINISHED r2_llama_lorma_central_dec | [08:31:46][r2_llama_lorma_central_dec] DONE in 2.2 min
FINISHED r2_gpt2_lorma_lead_dec | [08:32:04][r2_gpt2_lorma_lead_dec] DONE in 2.5 min
FINISHED r2_shared_full_lead_dec | [08:33:47][r2_shared_full_lead_dec] DONE in 1.7 min
FINISHED r2_encdec_lorma_lead_dec | [08:34:35][r2_encdec_lorma_lead_dec] DONE in 2.8 min
[ensemble] 22 candidate runs: r2_llama_lorma_central_dec=0.8420, dec_lorma_len256=0.8388, sm_llama_lorma_central=0.8371, dec_lorma_len128=0.8370, dec_lorma_len384=0.8363, sm_llama_lorma_leadcentral=0.8359, dec_lorma_nr3=0.8355, dec_lorma_free=0.8348, sm_llama_houlsby_lead=0.8343, sm_llama_full_lead=0.8333, sm_l